In [1]:
using Pkg
Pkg.activate("..")
include("../src/usadel_core.jl")
using Plots
using SparseArrays
using LinearAlgebra

  Activating project at `~/dev/julia/usadel_1D`


In [2]:
#parameters
Ln,Ls=200.0,100.0
dx=1.0
Γin=1e-3
σn,σs=1.0,1.0
p=setup_simulation(Ln,Ls,dx,0.0,Γin,σn,σs)
#checking
println("i=1", p.nodes[1])
ξn = sqrt(0.0152 / 2.0)
x_eval = -1.5 * ξn
println("evaluating DOS at x = $x_eval, ξn = $ξn")

energies = collect(range(0.001, 1.5, length=400))

i=1vacc
evaluating DOS at x = -0.1307669683062202, ξn = 0.08717797887081347


400-element Vector{Float64}:
 0.001
 0.004756892230576441
 0.008513784461152881
 0.012270676691729323
 0.016027568922305765
 0.019784461152882207
 0.023541353383458648
 0.02729824561403509
 0.03105513784461153
 0.03481203007518797
 0.03856892230576441
 0.04232581453634085
 0.046082706766917295
 ⋮
 1.4586741854636591
 1.4624310776942355
 1.466187969924812
 1.4699448621553886
 1.473701754385965
 1.4774586466165414
 1.4812155388471178
 1.4849724310776942
 1.4887293233082706
 1.4924862155388472
 1.4962431077694236
 1.5

In [3]:
theta_0=get_theta_0(p)
println(theta_0[p.N-1])

1.5697963271282298 + 0.0im


In [4]:

# γ = σn*ξs / σs*ξn  →  vary σn/σs ratio to get different γ
# ξs = sqrt(Ds/2Δ)
ξs = sqrt(0.00636 / 2.0)

# γ values from the paper: 0, 0.2, 1, 5
# γ = (σn/σs) * (ξs/ξn)  →  σn/σs = γ * ξn/ξs
γ_vals = [0.001, 0.2, 1.0, 5.0]   # 0.001 instead of 0 to avoid σn=0

plt = plot(xlabel="E/Δ", ylabel="N(E)/N₀",
           title="DOS in N metal at x=1.5ξn",
           ylims=(0, 1.5), xlims=(0, 1.2))

for γ in γ_vals
    σs = 1.0
    σn = γ * ξn / ξs

    p = setup_simulation(Ln, Ls, dx, 0.0, Γin, σn, σs)
    dos, idx = compute_DOS(energies, p, x_eval)

    plot!(plt, energies, dos, label="γ=$γ", lw=2)
end

display(plt)

LoadError: MethodError: no method matching !(::ComplexF64)
The function `!` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  !([91m::Missing[39m)
[0m[90m   @[39m [90mBase[39m [90m[4mmissing.jl:101[24m[39m
[0m  !([91m::Bool[39m)
[0m[90m   @[39m [90mBase[39m [90m[4mbool.jl:37[24m[39m
[0m  !([91m::ComposedFunction{typeof(!)}[39m)
[0m[90m   @[39m [90mBase[39m [90m[4moperators.jl:1154[24m[39m
[0m  ...


In [10]:
p = setup_simulation(200.0, 100.0, 1.0, 0.0, 1e-3, 1.0, 1.0)

p_test = params(0.5, p.σn, p.σs, p.Γin, p.D, p.Δ,
                p.dx, p.N, p.i0L, p.i0R,
                p.Ln, p.Ls, p.nodes)

theta_0 = get_theta_0(p_test)

println("N = ", p.N)
println("i0L = ", p.i0L, " → node: ", p.nodes[p.i0L])
println("i0R = ", p.i0R, " → node: ", p.nodes[p.i0R])
println("node[1] = ", p.nodes[1])
println("node[N] = ", p.nodes[p.N])
theta_0 = get_theta_0(p_test)
println("theta_0[1] (vacc): ", theta_0[1])
println("theta_0[i0L] (NS): ", theta_0[p.i0L])
println("theta_0[i0R] (SN): ", theta_0[p.i0R])
println("theta_0[N] (bulk): ", theta_0[p.N])
p_test = params(0.5, p.σn, p.σs, p.Γin, p.D, p.Δ,
                p.dx, p.N, p.i0L, p.i0R, p.Ln, p.Ls, p.nodes)

J, r = build_eq_sys(theta_0, p_test)
println("max residual at initial guess: ", maximum(abs.(r)))
println("size of J: ", size(J))
println("is J square? ", size(J,1) == size(J,2))
println("any NaN in r? ", any(isnan.(r)))
println("any Inf in r? ", any(isinf.(r)))
theta, converged, iters = newton_basic(theta_0, p_test)
println("converged: ", converged)
println("iters: ", iters)
println("max(|theta|): ", maximum(abs.(theta)))
println("any NaN in theta? ", any(isnan.(theta)))

# Cell — Newton step by step
theta = copy(theta_0)
for k in 1:10
    J, r = build_eq_sys(theta, p_test)
    res = maximum(abs.(r))
    
    η      = 1e-8
    dtheta = (J + η * sparse(I, p.N, p.N)) \ (-r)
    
    println("iter $k: max|r|=$(round(res, sigdigits=4))  max|dtheta|=$(round(maximum(abs.(dtheta)), sigdigits=4))  max|theta|=$(round(maximum(abs.(theta)), sigdigits=4))")
    
    theta .+= 0.5 .* dtheta
end


N = 300
i0L = 200 → node: NS
i0R = 201 → node: SN
node[1] = vacc
node[N] = bulk
theta_0[1] (vacc): 0.0 + 0.0im
theta_0[i0L] (NS): 0.7847314974221381 + 0.2746526277235706im
theta_0[i0R] (SN): 0.7847314974221381 + 0.2746526277235706im
theta_0[N] (bulk): 1.5694629948442762 + 0.5493052554471411im
max residual at initial guess: 0.46461853763932864
size of J: (300, 300)
is J square? true
any NaN in r? false
any Inf in r? false
converged: 0.0 + 0.0im
iters: 0.0 + 0.0im
max(|theta|): 0.0
any NaN in theta? false
iter 1: max|r|=0.4646  max|dtheta|=0.6152  max|theta|=1.663
iter 2: max|r|=0.2688  max|dtheta|=0.5204  max|theta|=1.663
iter 3: max|r|=0.1633  max|dtheta|=0.5185  max|theta|=1.663
iter 4: max|r|=0.09838  max|dtheta|=0.2914  max|theta|=1.663
iter 5: max|r|=0.03783  max|dtheta|=0.07635  max|theta|=1.663
iter 6: max|r|=0.0181  max|dtheta|=0.0336  max|theta|=1.663
iter 7: max|r|=0.008888  max|dtheta|=0.01593  max|theta|=1.663
iter 8: max|r|=0.004407  max|dtheta|=0.007774  max|theta|=1.663
i

In [11]:

J, r = build_eq_sys(theta_0, p_test)

# find where the biggest residuals are
sorted_idx = sortperm(abs.(r), rev=true)
println("Top 5 residual locations:")
for i in sorted_idx[1:5]
    println("  i=$i  node=$(p.nodes[i])  |r|=$(abs(r[i]))")
end

i = 201
h = p.dx
theta = theta_0

dth_N = (theta[i]   - theta[i-1]) / h
dth_S = (theta[i+1] - theta[i+2]) / h
cosN  = cos(theta[i])
cosS  = cos(theta[i+1])

println("theta[i-1] = ", theta[i-1])
println("theta[i]   = ", theta[i])
println("theta[i+1] = ", theta[i+1])
println("theta[i+2] = ", theta[i+2])
println("dth_N = ", dth_N)
println("dth_S = ", dth_S)
println("cosN  = ", cosN)
println("cosS  = ", cosS)
println("r[SN] = ", p_test.σn * cosN * dth_N - p_test.σs * cosS * dth_S)
# check what r[201] actually is from build_eq_sys
J, r = build_eq_sys(theta_0, p_test)
println("r[200] NS = ", r[200])
println("r[201] SN = ", r[201])

Top 5 residual locations:
  i=201  node=SN  |r|=0.46461853763932864
  i=199  node=N  |r|=0.006318693151153698
  i=202  node=S  |r|=0.0026438742395617477
  i=203  node=S  |r|=7.177418381854039e-17
  i=204  node=S  |r|=7.177418381854039e-17
theta[i-1] = 0.7847314974221381 + 0.2746526277235706im
theta[i]   = 0.7847314974221381 + 0.2746526277235706im
theta[i+1] = 1.5694629948442762 + 0.5493052554471411im
theta[i+2] = 1.5694629948442762 + 0.5493052554471411im
dth_N = 0.0 + 0.0im
dth_S = 0.0 + 0.0im
cosN  = 0.7344339580106826 - 0.19652847042819876im
cosS  = 0.0015395979807791749 - 0.5773487295934696im
r[SN] = 0.0 + 0.0im
r[200] NS = 0.0 + 0.0im
r[201] SN = -0.42124363278499866 + 0.19602088500005665im
